## Full vs others

In this section I will mostly compare `Full` hierarchy's performance with `Minimal`'s, as a baseline, but I will also reflect on how `Semantic` compared to `Minimal` in the previous chapters.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import numpy as np
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy, BinaryTemporalHierarchy, FullTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/binary_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
bth = BinaryTemporalHierarchy()
fth = FullTemporalHierarchy()

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=fth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=bth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
])

nrmse_em, nrmse_diffs = em, diffs

### QQ plots of reconciled forecast errors

First I will look at QQ plots of the reconciled forecast errors of `Full` and `Minimal` hierarchy at daily granularity

In [ ]:
plots.plot_error_qq(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    MI(hierarchy=fth, forecast_type=RECONCILED_FORECASTS, granularity=DAYS),
                    outlier_threshold=0.995).show()

In this chart, the first several percentiles are red, meaning they favor `Minimal` hierarchy, unlike how it was with `Semantic`, where its was not fully red. In the later percentiles (blue section), there is a bigger gap opening than there were with `Semantic`, meaning those percentiles of `Full` errors, are smaller than `Semantic`'s

Next I will look at the same chart at monthly granularity

In [ ]:
plots.plot_error_qq(nrmse_em,
                    MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    MI(hierarchy=fth, forecast_type=RECONCILED_FORECASTS, granularity=MONTHS),
                    outlier_threshold=0.995).show()

In this chart only the first few percentiles are red, meaning only the smallest percentiles favor `Minimal` hierarchy. At the later percentiles, `Minimal` performs worse, with a bigger gap than there were with `Semantic`.

### QQ plots of error differences

Next I will look at QQ plots comparing the distributions of the error differences of `Minimal` and `Full` hierarchy errors at daily and monthly granularity.

In [ ]:
plots.plot_error_qq(nrmse_diffs,
                    MI(hierarchy=f'{mth} - {mth}', granularity=DAYS),
                    MI(hierarchy=f'{fth} - {mth}', granularity=DAYS),
                    outlier_threshold=0.995).show()

At daily granularity, the shape of this chart is similar to that of `Semantic`, with slight differences. At both negative and positive percentiles, there is a bigger gap, meaning bigger imrpovements (in case of improvements) and bigger worsenings (in case of worsening).

In [ ]:
plots.plot_error_qq(nrmse_diffs,
                    MI(hierarchy=f'{mth} - {mth}', granularity=MONTHS),
                    MI(hierarchy=f'{fth} - {mth}', granularity=MONTHS),
                    outlier_threshold=0.995).show()

At monthly granularity, the same overall shape can be seen as with `Semantic`, the only difference being the bigger gap at the positive changes side (meaning smaller worsenings with `Full` hierarchy).

### QQ plot of hierarchy differences

In this section I will analyse the error differences between `Minimal` hierarchy's and `Full` hierarchy's reconciled forecasts.

In [ ]:
hier_diffs = ErrorDifference(nrmse_em,
                MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS),
                MI(hierarchy=fth, forecast_type=RECONCILED_FORECASTS))

In [ ]:
hier_diffs.get_error_stats(df_filter=lambda df: df[(df < np.quantile(df.values.flatten(), 0.999)) & (df > np.quantile(df.values.flatten(), 0.001))])

Looking at the statistics of these differences, we can see that the daily median is positive, meaning in more than half the cases, `Full` hierarchy performed worse (unlike how it was with `Semantic`). The mean of difference is smaller than it was with `Semantic`, at both granularities.

The negative error difference percentiles, at both granularities, are smaller for `Full`, and positive ones are smaller for `Semantic`

In [ ]:
plots.plot_error_qq(hier_diffs, MI(granularity=DAYS), MI(granularity=MONTHS), outlier_threshold=0.99)

The shape of this chart is very similar to how it was with `Semantic`, only on a different scale.

### Difference sign distributions

Next I will look at the distribution of the signs of these changes at the two granularities.

In [ ]:
plots.plot_error_signs(hier_diffs, MI(granularity=DAYS), MI(granularity=MONTHS), percentage=True)

Comparing this plot with how it was with `Semantic`:

- D-M- 42.64 vs 44.85
- D+M+ 33.65 vs 30.56
- D+M- 18.1 vs 17.68
- D-M+ 5.61 vs 6.91

- D- 48.25 vs 51.76
- M- 60.74 vs 62.53

We can deduce the amount of negative changes worsened going from `Semantic` to `Full`

### Full vs Semantic

Next I will compare the reconciled forecast errors of `Full` and `Semantic` directly, but also the differences of the two hierarchies to `Minimal` with each other.

In [ ]:
hier_diffs_fs = MergedErrorMetrics([
    ErrorDifference(nrmse_em,
                MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS),
                MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS)),
    ErrorDifference(nrmse_em,
                MI(hierarchy=mth, forecast_type=RECONCILED_FORECASTS),
                MI(hierarchy=fth, forecast_type=RECONCILED_FORECASTS))
])

hier_diffs_full_vs_sem = ErrorDifference(nrmse_em,
                MI(hierarchy=sth, forecast_type=RECONCILED_FORECASTS),
                MI(hierarchy=fth, forecast_type=RECONCILED_FORECASTS))


First I will plot the `Full` vs `Semantic` differences. `Semantic` is treated as the base here, so negative differences favor `Full` hierarchy

In [ ]:
plots.plot_error_signs(hier_diffs_full_vs_sem, MI(granularity=DAYS), MI(granularity=MONTHS), percentage=True)

Here we can see that in more than half cases, `Full` produced better monthly error, while daily difference sign is almost fiflty fifty. In 40% of the cases, switching hierarchies was a plain improvement.

In [ ]:
hier_diffs_full_vs_sem.get_error_stats(df_filter=lambda df: df[(df < np.quantile(df.values.flatten(), 0.999)) & (df > np.quantile(df.values.flatten(), 0.001))])

Looking at the error difference statistics, with both outlier ends removed, the mean of difference is negative in both granularities, meaning on average, `Full` produced slightly better reconciled forecasts. The median is positive in daily case (as we saw with signs), and negative in the monthly case.

Next I look at QQ plots comparing the distribution of `Full - Minimal` and `Semantic - Minimal` differences at daily and monthly granularity separately.

In [ ]:
plots.plot_error_qq(hier_diffs_fs,
                    MI(hierarchy=f'{sth} - {mth}', granularity=DAYS),
                    MI(hierarchy=f'{fth} - {mth}', granularity=DAYS),
                    outlier_threshold=0.995)

Here the plot is almost a straight line, with negative differences favoring `Full`, and positive differences favoring `Semantic`. This means improvements by reconciliation were magnified by having more aggregation levels, and degradations by reconciliation were also similarly magnified.

In [ ]:
plots.plot_error_qq(hier_diffs_fs,
                    MI(hierarchy=f'{sth} - {mth}', granularity=MONTHS),
                    MI(hierarchy=f'{fth} - {mth}', granularity=MONTHS),
                    outlier_threshold=0.995)

The same can be said at monthly granularity.

Next I will look at histograms comparing these two distributions

In [ ]:
plots.plot_error_distribution(hier_diffs_fs,
                    [MI(hierarchy=f'{sth} - {mth}', granularity=DAYS),
                     MI(hierarchy=f'{fth} - {mth}', granularity=DAYS),],
                    outlier_threshold=0.995)

Looking at the daily granularity reconciled forecast error differences, `Semantic` has more small daily negative differences, and `Full` has more daily bigger differences, both on negative and positive side.

In [ ]:
plots.plot_error_distribution(hier_diffs_fs,
                    [MI(hierarchy=f'{sth} - {mth}', granularity=MONTHS),
                     MI(hierarchy=f'{fth} - {mth}', granularity=MONTHS),],
                    outlier_threshold=0.995)

Looking at the monthly granularity reconciled forecast error differences, `Semantic` has more small differences (both positive and negative). `Full` has more big changes. This is further proven by the `Full - Minimal` differences having bigger variance values at both granularities than `Semantic - Minimal` differences had.

### Conclusions

**Full**
- When looking at QQ plots of reconciled forecast errors (`Full` vs `Minimal`), the first several percentiles favor `Minimal` at daily granularity, while the later percentiles show a bigger gap favoring `Full` than `Semantic` had. At monthly granularity, only the first few percentiles favor `Minimal`, with a larger gap favoring `Full` at later percentiles than was seen with `Semantic`.
- When looking at QQ plots of error differences (`Full` vs `Semantic`), the plots are almost straight lines with higher steepness than the identity, meaning improvements by reconciliation were magnified by having more aggregation levels, and degradations were similarly magnified.
- Pros
    - When looking at hierarchy difference statistics (`Full - Minimal`), the mean of difference is smaller than it was with `Semantic`, at both granularities.
    - When looking at `Full` vs `Semantic` error sign distributions, in 40% of cases switching from `Semantic` to `Full` was a strict improvement at both granularities, and in 55% of cases the monthly error improved.
    - When looking at `Full` vs `Semantic` error difference statistics, the mean is negative at both granularities, meaning `Full` produced slightly better reconciled forecasts on average.
- Cons
    - When looking at hierarchy difference statistics (`Full - Minimal`), the daily median is positive, meaning in more than half the cases `Full` performed worse (unlike `Semantic` which had a negative daily median).
    - When looking at hierarchy difference sign distributions (`Full - Minimal`), there were fewer negative-negative cases and fewer monthly negative changes than with `Semantic`.

Highlights:
- `Full` vs `Semantic` QQ plots (with previous chapter)
- `Full` vs `Semantic` reconciled error difference statistics
- `Full` vs `Minimal` reconciled error difference statistics